In [56]:
# Generic inputs for most ML tasks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
# This is new
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

pd.options.display.float_format = '{:,.2f}'.format

# setup interactive notebook mode
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from IPython.display import display, HTML

In [57]:
# fetch data 

mco_syr_sw_data = pd.read_csv('flight_data\mco_syr_sw_22_23.csv')
mco_syr_jb_data = pd.read_csv('flight_data\mco_syr_jb_22_23.csv')
jfk_syr_jb_data = pd.read_csv('flight_data\jfk_syr_jb_22_23.csv')
jfk_syr_end_data = pd.read_csv('flight_data\jfk_syr_end_22_23.csv')
ord_syr_ua_data = pd.read_csv('flight_data\ord_syr_ua_22_23.csv')
ord_syr_aa_data = pd.read_csv('flight_data\ord_syr_aa_22_23.csv')
# mco_syr_sw_data.head()
# mco_syr_jb_data.head()
# jfk_syr_jb_data.head()
# jfk_syr_end_data.head()
# ord_syr_ua_data.head()
# ord_syr_aa_data.head()

In [58]:
dfs = [ord_syr_aa_data,ord_syr_ua_data,jfk_syr_end_data,jfk_syr_jb_data,mco_syr_jb_data,mco_syr_sw_data]
main_data = pd.concat(dfs,axis = 0)
main_data.head()
len(main_data)

,Unnamed: 0,Carrier_Code,Date,Flight_Number,Tail_Number,Destination_Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),...,Wheels-on Time,Taxi-In time (Minutes),arr_Delay_Carrier,arr_Delay_Weather,arr_Delay_National_Aviation_System,arr_Delay_Security,arr_Delay_Late_Aircraft_Arrival,arr_hour,arr_day,arr_year
0,0,MQ,2022-01-01,4316,N900AE,SYR,15:28,0:00,107,0,...,0:00,0,0,0,0,0,0,18,5,2022
1,1,MQ,2023-01-01,3392,N283NN,SYR,18:25,18:24,111,104,...,21:02,6,0,0,0,0,0,21,6,2023
2,2,MQ,2023-01-01,3518,N213NN,SYR,13:05,13:01,117,96,...,15:33,4,0,0,0,0,0,16,6,2023
3,3,MQ,2022-01-02,4134,N900AE,SYR,10:25,10:21,111,146,...,13:43,4,0,0,31,0,0,13,6,2022
4,4,MQ,2022-01-02,4253,N238NN,SYR,17:25,17:51,107,116,...,20:40,7,0,26,9,0,0,20,6,2022


5623

In [59]:
main_data.dtypes



Unnamed: 0                             int64
Carrier_Code                          object
Date                                  object
Flight_Number                          int64
Tail_Number                           object
Destination_Airport                   object
Scheduled departure time              object
Actual departure time                 object
Scheduled elapsed time (Minutes)       int64
Actual elapsed time (Minutes)          int64
Departure delay (Minutes)              int64
Wheels-off time                       object
Taxi-Out time (Minutes)                int64
dep_Delay_Carrier                      int64
dep_Delay_Weather                      int64
dep_Delay_National_Aviation_System     int64
dep_Delay_Security                     int64
dep_Delay_Late_Aircraft_Arrival        int64
dep_hour                               int64
dep_day                                int64
dep_year                               int64
dep_order                             object
Origin_Air

In [60]:
# main_data['dep_min'] = main_data['Scheduled departure time'].str.split(":").str[1].astype('int64')
# main_data['Date'] = pd.to_datetime( main_data['Date'],format ="%Y-%m-%d") 
# main_data['Scheduled departure time'] = pd.to_datetime( main_data['Scheduled departure time'],format ='%H:%M') 
# main_data['Actual Arrival Time'] = pd.to_datetime( main_data['Actual Arrival Time'],format ='%H:%M')
# #main_data['dep_day'] = main_data['Date'].dt.day_of_week.astype('int64') 
# #main_data['dep_year'] = main_data['Date'].dt.year.astype('int64')
# main_data.dtypes

In [61]:
sub_data = main_data.drop(columns = ['Unnamed: 0','Destination_Airport','Actual departure time','Scheduled elapsed time (Minutes)','Actual elapsed time (Minutes)',
                         'Departure delay (Minutes)','Wheels-off time','Taxi-Out time (Minutes)','dep_Delay_Carrier',
                         'dep_Delay_Weather','dep_Delay_National_Aviation_System','dep_Delay_Security','dep_Delay_Late_Aircraft_Arrival',
                         'dep_year','Actual Arrival Time','Wheels-on Time','Taxi-In time (Minutes)','arr_Delay_Carrier','arr_Delay_Weather',
                         'arr_Delay_National_Aviation_System','arr_Delay_Security','arr_Delay_Late_Aircraft_Arrival','arr_year'])

In [62]:
sub_data.dtypes


Carrier_Code                object
Date                        object
Flight_Number                int64
Tail_Number                 object
Scheduled departure time    object
dep_hour                     int64
dep_day                      int64
dep_order                   object
Origin_Airport              object
Scheduled Arrival Time      object
Arrival Delay (Minutes)      int64
arr_hour                     int64
arr_day                      int64
dtype: object

In [63]:
sub_data.head()

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,Arrival Delay (Minutes),arr_hour,arr_day
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,latter,ORD,18:15,0,18,5
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,latter,ORD,21:16,-8,21,6
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,latter,ORD,16:02,-25,16,6
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,latter,ORD,13:16,31,13,6
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,latter,ORD,20:12,35,20,6


In [64]:
sub_data['dep_min'] = sub_data['Scheduled departure time'].str.split(":").str[1].astype('int64')
def round_to_nearest_quarter(minute):
    if 0 <= minute < 14:
        return 0
    elif 15 <= minute < 29:
        return 15
    elif 30 <= minute < 44:
        return 30
    else:
        return 45

sub_data['dep_minutes'] = sub_data['dep_min'].apply(round_to_nearest_quarter).astype('object')
sub_data['dep_hours'] = np.where(sub_data['dep_min']>52,sub_data['dep_hour']+1,sub_data['dep_hour']).astype('object')
sub_data['dep_min'] = sub_data['dep_min'].astype('object')
sub_data['arr_min'] = main_data['Scheduled Arrival Time'].str.split(":").str[1].astype('int64')
sub_data['arr_minutes'] = sub_data['arr_min'].apply(round_to_nearest_quarter).astype('object')
sub_data['arr_hours'] = np.where(sub_data['arr_min']>52,sub_data['arr_hour']+1,sub_data['arr_hour']).astype('object')
sub_data['arr_min'] = sub_data['arr_min'].astype('object')
sub_data['Flight_Number'] = main_data['Flight_Number'].astype('object')
sub_data['dep_hour'] = main_data['dep_hour'].astype('object')
sub_data['dep_day'] = main_data['dep_day'].astype('object')
sub_data['arr_hour'] = main_data['arr_hour'].astype('object')
sub_data['arr_day'] = main_data['arr_day'].astype('object')
conditions = [
    (sub_data['Arrival Delay (Minutes)'] > 5),
    (sub_data['Arrival Delay (Minutes)'] >=-5) & (sub_data['Arrival Delay (Minutes)'] <= 5),
    (sub_data['Arrival Delay (Minutes)'] < -5)
]
choices = [2,1,0]
sub_data['arr_status'] = np.select(conditions, choices)
sub_data.dtypes
sub_data.head()

Carrier_Code                object
Date                        object
Flight_Number               object
Tail_Number                 object
Scheduled departure time    object
dep_hour                    object
dep_day                     object
dep_order                   object
Origin_Airport              object
Scheduled Arrival Time      object
Arrival Delay (Minutes)      int64
arr_hour                    object
arr_day                     object
dep_min                     object
dep_minutes                 object
dep_hours                   object
arr_min                     object
arr_minutes                 object
arr_hours                   object
arr_status                   int32
dtype: object

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,Arrival Delay (Minutes),arr_hour,arr_day,dep_min,dep_minutes,dep_hours,arr_min,arr_minutes,arr_hours,arr_status
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,latter,ORD,18:15,0,18,5,28,15,15,15,15,18,1
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,latter,ORD,21:16,-8,21,6,25,15,18,16,15,21,0
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,latter,ORD,16:02,-25,16,6,5,0,13,2,0,16,0
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,latter,ORD,13:16,31,13,6,25,15,10,16,15,13,2
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,latter,ORD,20:12,35,20,6,25,15,17,12,0,20,2


In [65]:
sub_data.drop(columns= ['Date','Arrival Delay (Minutes)','Flight_Number','Tail_Number',
                       'Scheduled departure time','Scheduled Arrival Time','dep_order'],inplace = True)


In [32]:

sub_data.dtypes

sub_data.head()

Carrier_Code      object
dep_hour          object
dep_day           object
Origin_Airport    object
arr_hour          object
arr_day           object
dep_min           object
arr_min           object
arr_status         int32
dtype: object

,Carrier_Code,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min,arr_status
0,MQ,15,5,ORD,18,5,28,15,1
1,MQ,18,6,ORD,21,6,25,16,0
2,MQ,13,6,ORD,16,6,05,02,0
3,MQ,10,6,ORD,13,6,25,16,2
4,MQ,17,6,ORD,20,6,25,12,2


In [12]:
sub_data['Carrier_Code'].unique()

array(['MQ', 'UA', '9E', 'B6', 'WN'], dtype=object)

In [33]:
sub_data["Carrier_Code"] = np.where(sub_data["Carrier_Code"] == "MQ", "AA", sub_data["Carrier_Code"])
sub_data["Carrier_Code"] = np.where(sub_data["Carrier_Code"] == "9E", "DL", sub_data["Carrier_Code"])

In [34]:
sub_data['Origin_Airport'].unique()

array(['ORD', 'JFK', 'MCO'], dtype=object)

In [35]:
su_data = sub_data
# su_data.head()
# sub_data.columns
su_data['dep_hour'] = pd.Categorical(su_data['dep_hour'], categories=[i for i in range(24)])
su_data['dep_day'] = pd.Categorical(su_data['dep_day'], categories=[i for i in range(7)])
su_data['dep_min'] = pd.Categorical(su_data['dep_min'], categories=[i for i in range(60)])
su_data['arr_hour'] = pd.Categorical(su_data['arr_hour'], categories=[i for i in range(24)])
su_data['arr_day'] = pd.Categorical(su_data['arr_day'], categories=[i for i in range(7)])
su_data['arr_min'] = pd.Categorical(su_data['arr_min'], categories=[i for i in range(60)])
su_data['Carrier_Code'] = pd.Categorical(su_data['Carrier_Code'], categories=['AA', 'UA', 'DL', 'B6', 'WN'])
su_data['Origin_Airport'] = pd.Categorical(su_data['Origin_Airport'], categories=['ORD', 'JFK', 'MCO'])
su_data.columns

Index(['Carrier_Code', 'dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour',
       'arr_day', 'dep_min', 'arr_min', 'arr_status'],
      dtype='object')

In [36]:
su_data = pd.get_dummies(su_data, drop_first = True)
#su_data['arr_status'] = su_data['arr_status'].astype('object')
su_data.head()
su_data.dtypes
su_data.columns

,arr_status,Carrier_Code_UA,Carrier_Code_DL,Carrier_Code_B6,Carrier_Code_WN,dep_hour_1,dep_hour_2,dep_hour_3,dep_hour_4,dep_hour_5,...,arr_min_50,arr_min_51,arr_min_52,arr_min_53,arr_min_54,arr_min_55,arr_min_56,arr_min_57,arr_min_58,arr_min_59
0,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


arr_status         int32
Carrier_Code_UA     bool
Carrier_Code_DL     bool
Carrier_Code_B6     bool
Carrier_Code_WN     bool
                   ...  
arr_min_55          bool
arr_min_56          bool
arr_min_57          bool
arr_min_58          bool
arr_min_59          bool
Length: 183, dtype: object

Index(['arr_status', 'Carrier_Code_UA', 'Carrier_Code_DL', 'Carrier_Code_B6',
       'Carrier_Code_WN', 'dep_hour_1', 'dep_hour_2', 'dep_hour_3',
       'dep_hour_4', 'dep_hour_5',
       ...
       'arr_min_50', 'arr_min_51', 'arr_min_52', 'arr_min_53', 'arr_min_54',
       'arr_min_55', 'arr_min_56', 'arr_min_57', 'arr_min_58', 'arr_min_59'],
      dtype='object', length=183)

In [37]:
X_train, X_test, y_train, y_test = train_test_split(su_data.drop(columns = ['arr_status']), su_data['arr_status'], test_size=0.2, stratify = su_data['arr_status'], random_state=50)
# In the above split the stratify = y essentially makes sure the fractions of the classification is maintained
X_train
X_test
y_train.dtypes
y_test

,Carrier_Code_UA,Carrier_Code_DL,Carrier_Code_B6,Carrier_Code_WN,dep_hour_1,dep_hour_2,dep_hour_3,dep_hour_4,dep_hour_5,dep_hour_6,...,arr_min_50,arr_min_51,arr_min_52,arr_min_53,arr_min_54,arr_min_55,arr_min_56,arr_min_57,arr_min_58,arr_min_59
43,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1052,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
451,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
321,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
60,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
706,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
519,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
44,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


,Carrier_Code_UA,Carrier_Code_DL,Carrier_Code_B6,Carrier_Code_WN,dep_hour_1,dep_hour_2,dep_hour_3,dep_hour_4,dep_hour_5,dep_hour_6,...,arr_min_50,arr_min_51,arr_min_52,arr_min_53,arr_min_54,arr_min_55,arr_min_56,arr_min_57,arr_min_58,arr_min_59
136,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1214,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1187,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
675,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
175,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
481,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
149,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
128,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
474,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


dtype('int32')

136     2
1214    2
1187    0
675     0
175     0
       ..
481     0
149     0
128     2
474     1
608     1
Name: arr_status, Length: 1125, dtype: int32

In [38]:
#model = LogisticRegression(fit_intercept = True, solver='newton-cg', multi_class = 'multinomial', penalty = 'none', max_iter = 1000)
# model = LogisticRegression(fit_intercept = True, solver='lbfgs', multi_class = 'ovr', penalty = 'none', max_iter = 1000)
model = LogisticRegression(fit_intercept = True, solver='lbfgs', multi_class = 'multinomial', penalty = 'none', max_iter = 1000)
# model = LogisticRegression(fit_intercept = True, solver='newton-cg', multi_class = 'multinomial', penalty = 'none', max_iter = 10000)

# While using multiclass case do multi_class = 'ovr' or 'auto'; can also try other solvers
# While doing regularization, use penalty = 'l2' and also C = 10.0 (need to try other values too)

model.fit(X_train, y_train) 

# The following gives the mean accuracy on the given data and labels
model.score(X_train, y_train) 

# This is the coefficient Beta_1, ..., Beta_7
model.coef_

# This is the coefficient Beta_0
model.intercept_

C:\Users\gurud\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(


LogisticRegression(max_iter=1000, multi_class='multinomial', penalty='none')

0.5188972876834148

array([[ 4.20480433e-01,  5.41242368e-01, -2.05300319e-01,
        -2.14695690e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  3.37997540e+00,  8.87082803e-01,
         3.29912608e-01,  2.10778543e-01, -2.90859927e-01,
        -1.37159728e-01, -5.03918694e-01, -2.59765561e-01,
        -6.70869343e-01, -5.09993304e-01, -1.26640280e+00,
        -1.26782285e+00, -1.06618462e+00, -1.00893986e+00,
        -1.06349580e+00, -1.40234751e+00, -2.37981680e+00,
         7.43780177e-02,  1.17431842e-01,  8.51737647e-02,
        -1.72173231e-02,  3.23571753e-02,  2.94364436e-02,
         1.59041573e-01, -3.77952137e-02,  0.00000000e+00,
         4.17111122e+00,  4.17111122e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00, -2.23574436e+00, -1.56212919e+00,
        -1.51392818e+00, -1.73371143e+00, -9.05173836e-01,
        -9.02490138e-01, -5.30794728e-01, -7.78088253e-0

array([ 1.322395  , -0.95670851, -0.36568649])

In [39]:
model.score(X_test,y_test)

0.5004444444444445

In [40]:

from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=50, min_samples_split = 12, min_samples_leaf = 6, max_depth = 4, n_estimators = 300)

gb = gb.fit(X_train, y_train) 
gb.score(X_train, y_train) 

0.5573588261449534

In [41]:

gb.score(X_test,y_test)

0.5022222222222222